# 06a · Object Detection Fundamentals `[local]`

**Goal:** understand how detectors find *what* and *where*: boxes, IoU, anchors, NMS, the main detector families, and **mAP**, the metric everyone reports and few can compute. You'll build all of it from scratch, check it against the official tools, then use it to compare real pretrained detectors honestly.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | From classification to detection; two-stage vs. one-stage; why NMS exists |
| 📐 Math | Box formats, IoU, box encoding, anchors, focal loss, FPN, AP and mAP step by step |
| 🔨 From scratch | IoU, NMS, anchors, box encode/decode, **COCO-style mAP** (checked against `pycocotools`) |
| 🚀 Real detectors | Faster R-CNN (see its proposals!), YOLO11, YOLO26, all scored on VOC with no training |
| 📊 Evaluate | mAP table with speed, AP by object size, PR curves, error breakdown, choosing a score threshold |
| 🧪 Experiment | NMS threshold, input resolution, your NMS vs. the library's |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import io
import os
import time
import contextlib
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
from PIL import Image
from cvlib.datasets import voc2007_detection, VOC_CLASSES, VOC_TO_COCO
from cvlib.detection import box_convert, box_iou, nms, batched_nms, evaluate_detections, to_coco
from cvlib.data import model_path
from cvlib.viz import show

os.environ["YOLO_VERBOSE"] = "False"
device = env.device
rng = np.random.default_rng(0)

def sync():
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

---
# 🧠 1 · Intuition

### From "what" to "what and where"
A classifier says *"there's a dog"*. A detector says *"dog at (48, 110)–(301, 395), 94% sure; person at …"*, for **every** object, **any number** of them. That's two jobs per object: **classify** it and **regress** its box (4 numbers).

### The naive way (and why it's slow)
Slide a classifier over every position and size (module 01d's HOG detector). Millions of windows per image. Modern detectors instead share one CNN backbone across the whole image and predict boxes **densely** from its feature maps.

### Two families
- **Two-stage (Faster R-CNN):** stage 1, a **Region Proposal Network**, proposes ~1,000 "something might be here" boxes. Stage 2 crops each proposal's features (**RoIAlign**) and classifies + refines it. Accurate, slower.
- **One-stage (YOLO, SSD, RetinaNet, FCOS):** predict class + box **directly** at every location of the feature map in one pass. Fast, now just as accurate.

### Anchors: starting guesses
Instead of predicting a box from nothing, many detectors start from **anchors**: preset boxes of several sizes and shapes tiled over the image. The network predicts small **corrections** to the anchor ("shift a bit right, make it taller") plus a class score. *Anchor-free* detectors (FCOS, YOLOv8+) instead predict, at each location, the distances to the four box edges.

### NMS: one object, one box
Dense prediction means many neighboring locations all "see" the same dog and each predicts a box. **Non-maximum suppression** keeps the most confident box and deletes the others that overlap it heavily. (DETR and YOLO26 are designed to not need NMS: modules 09 and below.)

### How do we score a detector?
A prediction is correct if it has the right class **and** overlaps a real object enough (**IoU** ≥ threshold). Sweep the confidence threshold → precision/recall curve → **AP** per class → mean over classes = **mAP**. Section 2 walks through it step by step.

---
# 📐 2 · The math

### Box formats (the #1 source of detection bugs)

| Format | Numbers | Used by |
|---|---|---|
| `xyxy` | $x_1, y_1, x_2, y_2$ (corners) | torchvision, most math |
| `xywh` | $x_1, y_1, w, h$ (top-left + size) | COCO JSON |
| `cxcywh` | $c_x, c_y, w, h$ (center + size) | YOLO files (normalized to 0–1 by image size), DETR |

### IoU (Intersection over Union)
$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
1 = identical, 0 = no overlap. Detection uses IoU both to decide "correct?" and inside NMS.

### Box encoding (how regression targets are made)
Relative to an anchor $(x_a, y_a, w_a, h_a)$ (center format), a ground-truth box $(x, y, w, h)$ becomes:
$$t_x = \frac{x - x_a}{w_a},\quad t_y = \frac{y - y_a}{h_a},\quad t_w = \log\frac{w}{w_a},\quad t_h = \log\frac{h}{h_a}$$
Dividing by anchor size makes the targets scale-independent; the log makes "twice as wide" and "half as wide" symmetric. The network predicts $t$; decoding inverts it.

### Feature Pyramid Network (FPN)
Small objects need high-resolution features; large objects need deep, semantic ones. FPN takes the backbone's stages and adds a **top-down path with lateral connections**, so *every* resolution gets semantically strong features. Small objects are detected on the fine levels, large ones on the coarse levels.

### Focal loss (RetinaNet)
A one-stage detector scores ~100k locations per image, nearly all easy background. Plain cross-entropy is dominated by those easy negatives. Focal loss $-(1-p_t)^\gamma \log p_t$ (you built it in 05b) down-weights them: this is what made one-stage detectors as accurate as two-stage ones.

### NMS (greedy)
1. sort boxes by score
2. keep the top one; delete every remaining box with IoU > threshold (typically 0.5–0.7) with it
3. repeat with what's left. Done per class.

### AP and mAP, step by step
For **one class** at **one IoU threshold** $t$:
1. Collect all predictions of that class across the dataset; sort by score, highest first.
2. Walk down the list. A prediction is a **TP** if it overlaps a not-yet-matched ground-truth box of that class with IoU ≥ $t$; otherwise an **FP** (duplicates of an already-matched object are FPs too).
3. After each prediction compute precision = TP / (TP + FP) and recall = TP / (#ground-truth boxes): the **PR curve**.
4. Smooth the curve so precision never increases with recall, and average it over 101 recall points (0, 0.01, …, 1): that's **AP**.

Then:
- **AP50** (VOC metric): mean AP over classes at $t = 0.5$
- **mAP** or **AP@[.50:.95]** (COCO primary metric): also average over $t = 0.50, 0.55, \dots, 0.95$, which rewards **precise** boxes
- **AP_small / medium / large**: only objects with area < 32², between, > 96² pixels

Boxes marked *difficult* (VOC) or *crowd* (COCO) are **ignored**: matching them earns nothing, and missing them costs nothing.

---
# 🔨 3 · From scratch

### 3.1 · Box formats and IoU

The functions live in `cvlib/detection.py`. Read it alongside this section: every function is short.

In [ ]:
a = np.array([[50, 50, 150, 150]], float)
b = np.array([[100, 100, 200, 200], [50, 50, 150, 150], [300, 300, 350, 350]], float)
print("IoU:", box_iou(a, b).round(3), " (partial, identical, none)")
print("xyxy → xywh  :", box_convert(a, "xyxy", "xywh"))
print("xyxy → cxcywh:", box_convert(a, "xyxy", "cxcywh"))
print("round trip ok:", np.allclose(box_convert(box_convert(b, "xyxy", "cxcywh"), "cxcywh", "xyxy"), b))

vis = np.full((250, 400, 3), 255, np.uint8)
for (x1, y1, x2, y2), col in zip([a[0], b[0]], [(255, 0, 0), (0, 0, 255)]):
    cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
cv2.rectangle(vis, (100, 100), (150, 150), (0, 200, 0), -1)
show(vis, f"IoU = intersection (green) / union = {box_iou(a, b[:1])[0, 0]:.3f}", size=4)

A shift of half the box width already drops IoU to 0.14. That's why IoU ≥ 0.5 is a meaningful bar, and ≥ 0.75 a strict one.

### 3.2 · Anchors and box encoding

In [ ]:
def make_anchors(img_h, img_w, stride, scales=(64, 128, 256), ratios=(0.5, 1.0, 2.0)):
    """One anchor per (scale, ratio) at every `stride` pixels. Returns (N, 4) xyxy."""
    cy, cx = np.meshgrid(np.arange(stride / 2, img_h, stride), np.arange(stride / 2, img_w, stride), indexing="ij")
    centers = np.stack([cx.ravel(), cy.ravel()], 1)
    shapes = np.array([[s * np.sqrt(1 / r), s * np.sqrt(r)] for s in scales for r in ratios])   # (w, h), area = s²
    c = np.repeat(centers, len(shapes), 0); wh = np.tile(shapes, (len(centers), 1))
    return box_convert(np.c_[c, wh], "cxcywh", "xyxy")

def encode(gt, anc):
    g, a_ = box_convert(gt, "xyxy", "cxcywh"), box_convert(anc, "xyxy", "cxcywh")
    return np.c_[(g[:, 0] - a_[:, 0]) / a_[:, 2], (g[:, 1] - a_[:, 1]) / a_[:, 3], np.log(g[:, 2] / a_[:, 2]), np.log(g[:, 3] / a_[:, 3])]

def decode(t, anc):
    a_ = box_convert(anc, "xyxy", "cxcywh")
    return box_convert(np.c_[t[:, 0] * a_[:, 2] + a_[:, 0], t[:, 1] * a_[:, 3] + a_[:, 1], np.exp(t[:, 2]) * a_[:, 2], np.exp(t[:, 3]) * a_[:, 3]], "cxcywh", "xyxy")

val_items = voc2007_detection("val")
item = next(it for it in val_items if len(it["boxes"]) >= 2 and not it["difficult"].any())
img = cv2.imread(str(item["path"]))
H, W = img.shape[:2]
anchors = make_anchors(H, W, stride=32)
print(f"{W}×{H} image, stride 32 → {len(anchors):,} anchors (9 per location)")

ious = box_iou(item["boxes"], anchors)
best = ious.argmax(1)                                              # best-matching anchor per object
t = encode(item["boxes"], anchors[best])
print("regression targets for each object (tx, ty, tw, th):\n", t.round(3))
print("decode(encode(gt)) == gt:", np.allclose(decode(t, anchors[best]), item["boxes"]))

vis = img.copy()
mid = len(anchors) // 2 - (len(anchors) // 2) % 9
for x1, y1, x2, y2 in anchors[mid:mid + 9].astype(int):
    cv2.rectangle(vis, (x1, y1), (x2, y2), (200, 200, 200), 1)
for g, k in zip(item["boxes"].astype(int), best):
    cv2.rectangle(vis, tuple(g[:2]), tuple(g[2:]), (0, 200, 0), 3)
    cv2.rectangle(vis, tuple(anchors[k][:2].astype(int)), tuple(anchors[k][2:].astype(int)), (0, 0, 255), 2)
show(vis, "gray: the 9 anchors at one location | green: objects | red: best-matching anchor (IoU "
     + ", ".join(f"{ious[i, k]:.2f}" for i, k in enumerate(best)) + ")", size=7)

The network's job is just to predict these small, well-scaled corrections from the best anchor. During training, an anchor counts as a positive if its IoU with an object is high (e.g. ≥ 0.7 for the RPN), as negative if low (< 0.3), and is ignored in between.

### 3.3 · NMS

In [ ]:
def jitter_boxes(box, n, scale=12, seed=0):
    r = np.random.default_rng(seed)
    return np.clip(box + r.normal(0, scale, (n, 4)), 0, None), r.uniform(0.3, 0.95, n)

cand_b, cand_s = [], []
for k, g in enumerate(item["boxes"]):
    bb, ss = jitter_boxes(g, 15, seed=k)
    cand_b.append(bb); cand_s.append(ss)
cand_b, cand_s = np.concatenate(cand_b), np.concatenate(cand_s)
keep = nms(cand_b, cand_s, 0.5)
keep_tv = torchvision.ops.nms(torch.tensor(cand_b), torch.tensor(cand_s), 0.5).numpy()
print(f"{len(cand_b)} candidate boxes → {len(keep)} after NMS | same as torchvision: {np.array_equal(keep, keep_tv)}")

before, after = img.copy(), img.copy()
for bx in cand_b.astype(int):
    cv2.rectangle(before, tuple(bx[:2]), tuple(bx[2:]), (0, 0, 255), 1)
for bx in cand_b[keep].astype(int):
    cv2.rectangle(after, tuple(bx[:2]), tuple(bx[2:]), (0, 200, 0), 3)
show([before, after], ["dense predictions (simulated)", "after NMS (IoU 0.5)"], size=5)

### 3.4 · mAP, and checking it against `pycocotools`

`evaluate_detections` in `cvlib/detection.py` implements the step-by-step recipe from section 2, including COCO's matching rules, ignored boxes, the 101-point interpolation and the size ranges. Before trusting it, compare it with `pycocotools` (the official COCO evaluation code) on random detections.

In [ ]:
def coco_eval(gts, preds, n_classes):
    gt, dets = to_coco(gts, preds, n_classes)
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    with contextlib.redirect_stdout(io.StringIO()):                    # pycocotools prints a lot
        c = COCO(); c.dataset = gt; c.createIndex(); d = c.loadRes(dets)
        E = COCOeval(c, d, "bbox"); E.evaluate(); E.accumulate(); E.summarize()
    return dict(zip(["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"], E.stats[:6]))

def valid(b):                                                          # boxes must have x2 > x1 and y2 > y1!
    return np.c_[np.minimum(b[:, 0], b[:, 2]), np.minimum(b[:, 1], b[:, 3]), np.maximum(b[:, 0], b[:, 2]) + 1, np.maximum(b[:, 1], b[:, 3]) + 1]

r_ = np.random.default_rng(1)
toy_gts, toy_preds = [], []
for _ in range(80):
    n = r_.integers(1, 6); xy = r_.uniform(0, 400, (n, 2)); wh = r_.uniform(8, 200, (n, 2))
    gb = np.c_[xy, xy + wh]; gl = r_.integers(0, 3, n)
    pb = valid(np.r_[gb + r_.normal(0, 8, gb.shape), r_.uniform(0, 450, (4, 4))])
    pl = np.r_[np.where(r_.random(n) < 0.85, gl, (gl + 1) % 3), r_.integers(0, 3, 4)]
    toy_gts.append(dict(boxes=gb, labels=gl)); toy_preds.append(dict(boxes=pb, scores=r_.random(len(pb)), labels=pl))
ours = evaluate_detections(toy_gts, toy_preds, n_classes=3)
ref = coco_eval(toy_gts, toy_preds, 3)
for k in ref:
    print(f"{k:<10} ours {ours[k]:.4f} | pycocotools {ref[k]:.4f}")

Identical to many decimal places. While building this, the first comparison actually **disagreed**, because the random test boxes included some with $x_2 < x_1$. The two implementations handle such invalid boxes differently (pycocotools happily computes a positive "area" from negative width × negative height). Lesson: **validate your boxes** before evaluating; it's one of the most common bugs in detection pipelines.

---
# 🚀 4 · Real detectors, no training

COCO's 80 classes include all 20 VOC classes (under slightly different names, e.g. *sofa* = *couch*). So COCO-pretrained detectors can be scored on VOC directly, just like the Imagenette trick in module 04.

### 4.1 · Faster R-CNN: watch the two stages

In [ ]:
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights,
                                          fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights)
w_r50 = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
frcnn = fasterrcnn_resnet50_fpn_v2(weights=w_r50).eval().to(device)
COCO_NAMES = w_r50.meta["categories"]                                # index = torchvision label id

def to_tensor(path):
    return torchvision.transforms.functional.to_tensor(Image.open(path).convert("RGB"))

x = to_tensor(item["path"]).to(device)
with torch.no_grad():
    images, _ = frcnn.transform([x])                                  # resize + normalize
    feats = frcnn.backbone(images.tensors)                            # FPN: several resolutions
    proposals, _ = frcnn.rpn(images, feats)                           # stage 1
    dets = frcnn([x])[0]                                              # full model (stage 1 + 2 + NMS)
scale = np.array([W / images.image_sizes[0][1], H / images.image_sizes[0][0]] * 2)
props = proposals[0].cpu().numpy() * scale
print("FPN feature maps:", {k: tuple(v.shape[-2:]) for k, v in feats.items()})
print(f"stage 1 (RPN): {len(props)} proposals | stage 2 + NMS: {int((dets['scores'] > 0.5).sum())} detections with score > 0.5")

v1, v2 = img.copy(), img.copy()
for bx in props[:300].astype(int):
    cv2.rectangle(v1, tuple(bx[:2]), tuple(bx[2:]), (0, 140, 255), 1)
for bx, s, lab in zip(dets["boxes"].cpu().numpy().astype(int), dets["scores"].cpu().numpy(), dets["labels"].cpu().numpy()):
    if s > 0.5:
        cv2.rectangle(v2, tuple(bx[:2]), tuple(bx[2:]), (0, 200, 0), 2)
        cv2.putText(v2, f"{COCO_NAMES[lab]} {s:.2f}", (bx[0], max(bx[1] - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 2)
show([v1, v2], ["stage 1: top 300 region proposals", "stage 2: classified + refined + NMS"], size=5.5)

### 4.2 · Scoring five detectors on VOC 2007

300 validation images, no training. For mAP we keep low-confidence detections too (score ≥ 0.001): AP is computed over *all* thresholds, so dropping low-scoring boxes would cut the PR curve short.

| Model | Type | Notes |
|---|---|---|
| Faster R-CNN MobileNetV3-FPN | two-stage | light backbone |
| Faster R-CNN ResNet-50-FPN v2 | two-stage | strong, slow |
| YOLO11n / YOLO11s | one-stage, anchor-free | nano and small sizes; NMS |
| YOLO26n | one-stage, **NMS-free** | trained to output one box per object (the DETR idea, module 09) |

In [ ]:
from ultralytics import YOLO

N_EVAL = 300
eval_items = [val_items[i] for i in rng.choice(len(val_items), N_EVAL, replace=False)]
gts = [dict(boxes=it["boxes"], labels=it["labels"], ignore=it["difficult"]) for it in eval_items]
coco_to_voc = {COCO_NAMES.index(c): VOC_CLASSES.index(v) for v, c in VOC_TO_COCO.items()}

def run_torchvision(model, items, score_thresh=0.001):
    model.roi_heads.score_thresh = score_thresh
    model.roi_heads.detections_per_img = 100
    preds, times = [], []
    for it in items:
        x = to_tensor(it["path"]).to(device)
        sync(); t0 = time.perf_counter()
        with torch.no_grad():
            d = model([x])[0]
        sync(); times.append(time.perf_counter() - t0)
        lab = d["labels"].cpu().numpy(); keep = np.isin(lab, list(coco_to_voc))
        preds.append(dict(boxes=d["boxes"].cpu().numpy()[keep], scores=d["scores"].cpu().numpy()[keep],
                          labels=np.array([coco_to_voc[l] for l in lab[keep]], dtype=np.int64)))
    return preds, float(np.median(times[3:]) * 1000)

def run_yolo(weights, items, imgsz=640, conf=0.001):
    m = YOLO(str(model_path(weights)))
    yolo_to_voc = {k: VOC_CLASSES.index(v) for k, n in m.names.items() for v, c in VOC_TO_COCO.items() if n == c}
    preds, times = [], []
    for it in items:
        sync(); t0 = time.perf_counter()
        r = m.predict(str(it["path"]), imgsz=imgsz, conf=conf, device=device, verbose=False, max_det=100)[0]
        sync(); times.append(time.perf_counter() - t0)
        lab = r.boxes.cls.cpu().numpy().astype(int); keep = np.isin(lab, list(yolo_to_voc))
        preds.append(dict(boxes=r.boxes.xyxy.cpu().numpy()[keep], scores=r.boxes.conf.cpu().numpy()[keep],
                          labels=np.array([yolo_to_voc[l] for l in lab[keep]], dtype=np.int64)))
    return preds, float(np.median(times[3:]) * 1000)

frcnn_mobile = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT).eval().to(device)
detectors = {
    "FRCNN MobileNetV3": lambda: run_torchvision(frcnn_mobile, eval_items),
    "FRCNN R50-FPN v2": lambda: run_torchvision(frcnn, eval_items),
    "YOLO11n": lambda: run_yolo("yolo11n.pt", eval_items),
    "YOLO11s": lambda: run_yolo("yolo11s.pt", eval_items),
    "YOLO26n": lambda: run_yolo("yolo26n.pt", eval_items),
}
det_preds, det_res = {}, {}
for name, fn in detectors.items():
    t0 = time.perf_counter()
    det_preds[name], ms = fn()
    det_res[name] = evaluate_detections(gts, det_preds[name], n_classes=20) | {"ms": ms}
    print(f"{name:<18} done in {time.perf_counter() - t0:.0f}s")

---
# 📊 5 · Evaluate

### 5.1 · The comparison table

In [ ]:
print(f"{'model':<18}{'mAP':>7}{'AP50':>7}{'AP75':>7}{'AP_S':>7}{'AP_M':>7}{'AP_L':>7}{'ms/img':>9}")
for name, r in det_res.items():
    print(f"{name:<18}{r['mAP']:>7.3f}{r['AP50']:>7.3f}{r['AP75']:>7.3f}{r['AP_small']:>7.3f}{r['AP_medium']:>7.3f}{r['AP_large']:>7.3f}{r['ms']:>9.1f}")

# cross-check one model with pycocotools (without the ignored 'difficult' boxes, which pycocotools handles differently)
gts_nodiff = [dict(boxes=g["boxes"][~g["ignore"]], labels=g["labels"][~g["ignore"]]) for g in gts]
ref = coco_eval(gts_nodiff, det_preds["YOLO11n"], 20)
print(f"\ncheck, YOLO11n without difficult boxes: ours mAP {evaluate_detections(gts_nodiff, det_preds['YOLO11n'], 20)['mAP']:.4f} | pycocotools {ref['mAP']:.4f}")

**Reading the table:**
- **AP50 vs. mAP:** AP50 forgives sloppy boxes; mAP (averaged up to IoU 0.95) rewards tight ones. A model can rank differently on the two.
- **Small objects are hard for everyone**: AP_small is far below AP_large. Few pixels, little detail. Higher input resolution helps (Experiment 2).
- **Speed:** the YOLO models run in a fraction of the R50 Faster R-CNN's time on your Mac. Accuracy per millisecond is why one-stage detectors dominate in practice.
- These are pretrained COCO models scored on VOC with **no training**. Fine-tuning on VOC itself (06b) adds more.

### 5.2 · Per-class AP and PR curves

In [ ]:
best_name = max(det_res, key=lambda k: det_res[k]["mAP"])
r = det_res[best_name]
ap50 = np.array([r["per_class_AP50"][c] for c in range(20)])
order = np.argsort(ap50)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].barh([VOC_CLASSES[c] for c in order], ap50[order]); axes[0].set_xlabel("AP50"); axes[0].set_title(f"{best_name}: per-class AP50")
for c in (order[0], order[len(order) // 2], order[-1]):
    prec, rec = r["pr_curves"][c]
    axes[1].plot(rec, prec, label=f"{VOC_CLASSES[c]} (AP50 {ap50[c]:.2f})")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_title("PR curves at IoU 0.5: worst, middle, best class")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

A PR curve that ends early on the right means the model **never finds** some objects at any confidence. A curve that drops early means its **most confident** predictions already include mistakes.

### 5.3 · What kind of mistakes? An error breakdown

Not all false positives are equal. Using predictions with score ≥ 0.3 (a typical deployment threshold), classify each error:

| Error | Meaning | Fix direction |
|---|---|---|
| **localization** | right class, overlaps an object, but IoU < 0.5 | better box regression, higher resolution |
| **duplicate** | a second box on an already-detected object | NMS settings |
| **class confusion** | good box, wrong class | better classifier, more data |
| **background** | box on nothing | harder negatives, higher threshold |
| **missed** | a real object with no box | recall: resolution, data, lower threshold |

In [ ]:
def error_breakdown(gts, preds, score_thr=0.3, iou_thr=0.5):
    counts = dict(TP=0, localization=0, duplicate=0, **{"class confusion": 0}, background=0, missed=0)
    for g, p in zip(gts, preds):
        keep = ~g["ignore"]
        gb, gl = g["boxes"][keep], g["labels"][keep]
        sel = p["scores"] >= score_thr
        pb, ps, pl = p["boxes"][sel], p["scores"][sel], p["labels"][sel]
        matched = np.zeros(len(gb), bool)
        ious = box_iou(pb, gb) if len(gb) and len(pb) else np.zeros((len(pb), len(gb)))
        for i in np.argsort(-ps):
            same = gl == pl[i]
            iou_same = np.where(same, ious[i], -1); j = int(iou_same.argmax()) if len(gb) else -1
            if j >= 0 and iou_same[j] >= iou_thr and not matched[j]:
                counts["TP"] += 1; matched[j] = True
            elif j >= 0 and iou_same[j] >= iou_thr:
                counts["duplicate"] += 1
            elif len(gb) and np.where(~same, ious[i], -1).max() >= iou_thr:
                counts["class confusion"] += 1
            elif j >= 0 and iou_same[j] >= 0.1:
                counts["localization"] += 1
            else:
                counts["background"] += 1
        counts["missed"] += int((~matched).sum())
    return counts

print(f"{'model':<18}" + "".join(f"{k:>16}" for k in ["TP", "localization", "duplicate", "class confusion", "background", "missed"]))
for name in det_preds:
    c = error_breakdown(gts, det_preds[name])
    print(f"{name:<18}" + "".join(f"{v:>16}" for v in c.values()))

The breakdown says *what to fix*. A model with many "missed" and "localization" errors needs resolution or better small-object handling; one with many "background" errors needs a higher threshold or harder negative examples. Note how few **duplicates** there are for every model: NMS (or YOLO26's NMS-free training) is doing its job.

Also note the Faster R-CNN rows: far more **background** false positives at the same 0.3 threshold. That doesn't simply mean "worse model". Different detectors produce differently scaled scores, so **a score threshold doesn't transfer between models**. Choose it per model, on validation data (next section).

### 5.4 · Choosing a score threshold for deployment

mAP is threshold-free, but a deployed system must pick one. Sweep it and look at precision, recall and F1 over all classes:

In [ ]:
best_preds = det_preds[best_name]
thrs = np.linspace(0.05, 0.9, 18)
P, R = [], []
for t in thrs:
    c = error_breakdown(gts, best_preds, score_thr=t)
    fp = c["localization"] + c["duplicate"] + c["class confusion"] + c["background"]
    P.append(c["TP"] / max(c["TP"] + fp, 1)); R.append(c["TP"] / max(c["TP"] + c["missed"], 1))
P, R = np.array(P), np.array(R); F1 = 2 * P * R / np.maximum(P + R, 1e-9)
plt.figure(figsize=(7, 3.8))
plt.plot(thrs, P, label="precision"); plt.plot(thrs, R, label="recall"); plt.plot(thrs, F1, label="F1", lw=2)
plt.axvline(thrs[F1.argmax()], ls="--", color="gray"); plt.xlabel("score threshold"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"{best_name}: best F1 {F1.max():.3f} at threshold {thrs[F1.argmax()]:.2f}"); plt.show()

Pick the threshold from what errors cost you: a safety system (pedestrian detection) favors recall (low threshold); a system that alerts a human for every detection favors precision (high threshold). Choose it on validation data, as in 05c.

---
# 🧪 6 · Experiments: predict first, then run

**Experiment 1: the NMS threshold.** Faster R-CNN uses 0.5 by default. Predict mAP and the number of duplicates at 0.3, 0.5, 0.7 and 0.9.

In [ ]:
# Prediction: ...
sub = eval_items[:100]; sub_gts = gts[:100]
for thr in (0.3, 0.5, 0.7, 0.9):
    frcnn_mobile.roi_heads.nms_thresh = thr
    p, _ = run_torchvision(frcnn_mobile, sub)
    c = error_breakdown(sub_gts, p)
    print(f"NMS IoU {thr}: mAP {evaluate_detections(sub_gts, p, 20)['mAP']:.3f} | duplicates at score ≥ 0.3: {c['duplicate']}")
frcnn_mobile.roi_heads.nms_thresh = 0.5
# On VOC, 0.3–0.7 barely differ in mAP: few scenes have heavily overlapping objects of the same class.
# At 0.9 duplicates flood through and mAP drops sharply. In crowded scenes (people in a queue, parked cars)
# a low threshold would start suppressing real neighbors, so 0.5–0.7 is the usual compromise.

**Experiment 2: input resolution and small objects.** YOLO11n at 320, 640 and 960 pixels. Predict AP_small and speed.

In [ ]:
# Prediction: ...
for sz in (320, 640, 960):
    p, ms = run_yolo("yolo11n.pt", sub, imgsz=sz)
    rr = evaluate_detections(sub_gts, p, 20)
    print(f"imgsz {sz}: mAP {rr['mAP']:.3f} | AP_small {rr['AP_small']:.3f} | AP_large {rr['AP_large']:.3f} | {ms:.1f} ms/img")
# More pixels per object helps small objects most. Cost grows roughly with the number of pixels.
# Large objects can get *worse* at high resolution: they no longer fit the receptive fields the model was trained for.

**Experiment 3: your NMS vs. torchvision's.** Same results? How much faster is the C++/GPU version on 5,000 boxes?

In [ ]:
# Prediction: ...
bb = valid(rng.uniform(0, 1000, (5000, 4))); ss = rng.random(5000)
t0 = time.perf_counter(); k1 = nms(bb, ss, 0.5); t_mine = time.perf_counter() - t0
tb, ts = torch.tensor(bb, dtype=torch.float32), torch.tensor(ss, dtype=torch.float32)
t0 = time.perf_counter(); k2 = torchvision.ops.nms(tb, ts, 0.5).numpy(); t_tv = time.perf_counter() - t0
print(f"same kept boxes: {np.array_equal(k1, k2)} | yours {t_mine * 1000:.0f} ms | torchvision {t_tv * 1000:.1f} ms → {t_mine / t_tv:.0f}× faster")
# NMS is a sequential loop, hard to parallelize: on edge devices it can take a real share of total latency.
# That's a big motivation for NMS-free detectors (DETR, YOLO26) and for exporting NMS into TensorRT (module 15).

---
# 📝 7 · Recap

**Cheat sheet**
- Detection = classify + regress a box, for every object. Know your box format: `xyxy`, `xywh` (COCO), `cxcywh` (YOLO, normalized).
- **IoU** = overlap / union. Used to define "correct" and inside NMS.
- **Anchors** + encoded offsets $(t_x, t_y, t_w, t_h)$; anchor-free models predict distances to the edges instead.
- **Two-stage** (RPN proposals → RoIAlign → classify/refine) vs. **one-stage** (dense predictions in one pass). **FPN** handles multiple scales; **focal loss** handles the background flood.
- **NMS** removes duplicates; its IoU threshold trades duplicates vs. suppressed neighbors. NMS-free: DETR, YOLO26.
- **AP** = area under the interpolated PR curve for one class; **AP50** (VOC) vs. **mAP@[.5:.95]** (COCO); AP by object size.
- Validate boxes; cross-check your evaluator; break errors down by type; choose a deployment threshold on validation data.

**Interview-style questions**
1. Walk through how mAP@[.5:.95] is computed, from raw predictions to one number.
2. Why does mAP evaluation keep very low-confidence detections?
3. Two-stage vs. one-stage detectors: what are the trade-offs, and why did the gap close?
4. Why encode box widths with $\log(w / w_a)$?
5. What goes wrong with an NMS threshold of 0.3 in a crowded scene? With 0.9?
6. How does an FPN help with small objects?
7. Your detector has a high AP50 but low mAP. What does that tell you?
8. A detector misses many small objects. List three things you'd try.

**Next:** `06b_training_detectors.ipynb`: fine-tuning Faster R-CNN and training YOLO on your own data.